# DDPM — Denoising Diffusion Probabilistic Models (2020)

_Papers · Generative Models_

**Slowly add noise to data, then train a network to undo one noise step at a time — and you can generate new samples from pure noise.**

---

This notebook is a practice scaffold for the **DDPM — Denoising Diffusion Probabilistic Models (2020)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)

# --- 0. Sanity-check the worked example: Eq. 4 on a tiny 4-step schedule. ---
betas_demo = torch.tensor([0.1, 0.2, 0.3, 0.4])
alphas_demo = 1 - betas_demo
abar_demo = torch.cumprod(alphas_demo, 0)          # running product: alpha-bar
x0, eps, t = torch.tensor(2.0), torch.tensor(0.5), 2
ab = abar_demo[t - 1]                              # alpha-bar_2 = 0.72
xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps         # Eq. 4
print("alpha-bar:", [round(v, 4) for v in abar_demo.tolist()])   # [0.9, 0.72, 0.504, 0.3024]
print("worked x_t (t=2):", round(xt.item(), 6))                  # 1.961631


# --- 1. The forward noising schedule (built by hand). ---
T = 50
betas  = torch.linspace(1e-4, 0.02, T)             # small linear variance schedule
alphas = 1 - betas
abar   = torch.cumprod(alphas, 0)                  # alpha-bar_t = prod_{s<=t} alpha_s


# --- 2. A 2-D toy distribution: 8 Gaussian clusters on a ring (8 modes). ---
def sample_data(n):
    k = torch.randint(0, 8, (n,))
    ang = k.float() / 8 * 2 * math.pi
    centers = torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0
    return centers + torch.randn(n, 2) * 0.15


# --- 3. The noise predictor eps_theta(x_t, t): a small MLP. use_t toggles the ablation. ---
class Denoiser(nn.Module):
    def __init__(self, h=128, use_t=True):
        super().__init__()
        self.use_t = use_t
        self.net = nn.Sequential(nn.Linear(2 + 1, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, 2))
    def forward(self, x, t):
        te = (t.float() / T).unsqueeze(1)
        if not self.use_t:
            te = torch.zeros_like(te)              # ablation: hide the timestep
        return self.net(torch.cat([x, te], 1))


def train(use_t=True, steps=3000):
    torch.manual_seed(0)
    net = Denoiser(use_t=use_t)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for step in range(steps):
        x0  = sample_data(512)
        tb  = torch.randint(0, T, (512,))          # random timesteps (Alg. 1)
        eps = torch.randn_like(x0)                 # the noise to predict
        ab  = abar[tb].unsqueeze(1)
        xt  = ab.sqrt() * x0 + (1 - ab).sqrt() * eps   # Eq. 4: forward-noise in one shot
        loss = ((eps - net(xt, tb)) ** 2).mean()       # Eq. 14: L_simple (predict the noise)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 500 == 0:
            print(f"  step {step:4d}  loss {loss.item():.4f}")
    return net


# --- 4. Sampling (Algorithm 2): start from noise, denoise step by step. ---
@torch.no_grad()
def sample(net, n=500, snapshot_at=(40, 30, 20, 10, 0)):
    x = torch.randn(n, 2)                          # x_T ~ N(0, I): pure noise
    snaps = {}
    for t in reversed(range(T)):                   # t = T-1, ..., 0
        tb  = torch.full((n,), t, dtype=torch.long)
        eps = net(x, tb)
        a, ab = alphas[t], abar[t]
        mean = (1 / a.sqrt()) * (x - ((1 - a) / (1 - ab).sqrt()) * eps)   # Eq. 11 / Alg. 2
        x = mean + betas[t].sqrt() * torch.randn_like(x) if t > 0 else mean  # no noise at last step
        if t in snapshot_at:
            snaps[t] = x.clone()
    return x, snaps


print("TRAIN (with timestep):")
net = train(use_t=True)
samples, snaps = sample(net)

# distance of each sample to the nearest of the 8 cluster centers -> did it find the ring?
ang   = torch.arange(8).float() / 8 * 2 * math.pi
modes = torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0
d = torch.cdist(samples, modes).min(1).values
print("\nfrac of samples within 0.4 of a cluster:", round((d < 0.4).float().mean().item(), 3))
print("PRINT samples emerging from noise (mean radius should grow toward 2.0):")
for t in (40, 30, 20, 10, 0):
    r = snaps[t].norm(dim=1).mean().item()
    print(f"  t={t:2d}  mean radius {r:.3f}")
# Typical: ~0.94 of samples land on a cluster; radius climbs ~1.5 -> ~2.0 as noise is removed.
# (Our small run -- not the paper's reported number.)

# --- 5. ABLATION: drop the timestep input, retrain, resample. Samples get worse. ---
print("\nABLATION (no timestep input):")
net_no_t = train(use_t=False)
samp_no_t, _ = sample(net_no_t)
d2 = torch.cdist(samp_no_t, modes).min(1).values
print("frac within 0.4 of a cluster, NO timestep:", round((d2 < 0.4).float().mean().item(), 3))
# Lower fraction: without t the one network cannot tell which noise level it is undoing.

## Visualize the data & results

_Starting from pure noise, do the reverse (denoising) steps pull the cloud back onto the 8 data clusters?_

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)

T = 50
betas  = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
abar   = torch.cumprod(alphas, 0)

def sample_data(n):
    k = torch.randint(0, 8, (n,))
    ang = k.float() / 8 * 2 * math.pi
    return torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0 + torch.randn(n, 2) * 0.15

class Denoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, (t.float() / T).unsqueeze(1)], 1))

net = Denoiser(); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
for _ in range(3000):                              # train on Eq. 14
    x0 = sample_data(512); tb = torch.randint(0, T, (512,)); eps = torch.randn_like(x0)
    ab = abar[tb].unsqueeze(1)
    xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
    loss = ((eps - net(xt, tb)) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()                                   # Algorithm 2: sample, snapshot the cloud
def sample(n=500, snap=(49, 16, 0)):
    x = torch.randn(n, 2); snaps = {}
    for t in reversed(range(T)):
        tb = torch.full((n,), t, dtype=torch.long); e = net(x, tb)
        a, ab = alphas[t], abar[t]
        mean = (1 / a.sqrt()) * (x - ((1 - a) / (1 - ab).sqrt()) * e)
        x = mean + betas[t].sqrt() * torch.randn_like(x) if t > 0 else mean
        if t in snap: snaps[t] = x[:60].clone()
    return snaps

torch.manual_seed(7)
snaps = sample()
for t in (49, 16, 0):
    print(f"t={t:2d}", [[round(p[0].item(), 3), round(p[1].item(), 3)] for p in snaps[t][:5]], "...")
# t=49 -> diffuse blob; t=16 -> pulling toward the ring; t=0 -> seated on the 8 clusters.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your trained model samples cleanly onto the 8 clusters. Now remove the
            timestep input $t$ from the network (feed only $x_t$) and retrain everything else identically.
            What happens to the samples, and what does that show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change the network so it ignores $t$ (e.g. always pass a constant in place of the timestep feature); keep architecture width, schedule, optimizer, data, and steps the same. — _An honest ablation changes exactly one thing &mdash; the timestep conditioning &mdash; so any difference is attributable to it._
- Retrain and sample. The points no longer settle tightly onto the 8 clusters: they stay diffuse or smear between modes. — _The same noisy value $x_t$ needs a different correction depending on how noisy it is. Without $t$, $\epsilon_\theta$ cannot tell an early (slightly noisy) step from a late (very noisy) one, so it predicts a blurry average._
- Conclude that the timestep conditioning, not just the network capacity, is what lets one network invert the whole schedule. — _One denoiser must handle all noise levels; $t$ is how it knows which level it is at._

**Answer:** Samples degrade &mdash; the cloud fails to condense cleanly onto the 8 clusters and instead
                 stays spread out or smears between modes. Because the timestep is the only thing removed, this
                 isolates timestep conditioning as essential: a single network must invert every
                 noise level, and $t$ tells it which level it is undoing. (The CODE includes this ablation.)

</details>

**Problem 2.** Using the worked schedule $\beta = [0.1, 0.2, 0.3, 0.4]$, noise a point $x_0 = 1.0$ to timestep
            $t = 4$ with noise $\epsilon = -1.0$. What is $x_4$, and what does each part represent?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall $\bar\alpha_4 = 0.9 \times 0.8 \times 0.7 \times 0.6 = 0.3024$. — _$\bar\alpha_t$ is the running product of $\alpha_s = 1-\beta_s$; at $t=4$ it is the full product._
- Coefficients: $\sqrt{0.3024} \approx 0.5499$ and $\sqrt{1 - 0.3024} = \sqrt{0.6976} \approx 0.8352$. — _Eq. 4 weights the clean part by $\sqrt{\bar\alpha_t}$ and the noise by $\sqrt{1-\bar\alpha_t}$._
- Mix: $x_4 = 0.5499 \times 1.0 + 0.8352 \times (-1.0) = 0.5499 - 0.8352 = -0.2853$. — _Plugging into $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$._

**Answer:** $x_4 \approx -0.285$. The first piece $0.5499 \times 1.0 \approx 0.55$ is the surviving
                 clean signal (only ~$0.55$ of the original $1.0$ is left this deep into the schedule); the
                 second piece $0.8352 \times (-1.0) \approx -0.84$ is the injected noise, now the
                 dominant part &mdash; which is exactly why late-$t$ points look almost like pure noise.

</details>

**Problem 3.** Why is the network trained to predict the noise $\epsilon$ rather than the clean point
            $x_0$ directly? Both are recoverable from each other given $x_t$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note from Eq. 4 that $x_0 = (x_t - \sqrt{1-\bar\alpha_t}\,\epsilon)/\sqrt{\bar\alpha_t}$, so predicting $\epsilon$ and predicting $x_0$ are algebraically interchangeable. — _Given $x_t$ and the schedule, knowing one determines the other._
- Consider the target's scale across timesteps: $\epsilon$ is always unit-variance Gaussian noise, the same scale for every $t$. The implied $x_0$ correction is not. — _A constant-scale target is easier to regress; the loss is well-conditioned at every noise level._
- Recall the paper's empirical result: substituting the $\epsilon$ parameterization (Eq. 11) and the unweighted MSE (Eq. 14) gave better samples than the direct alternatives. — _Theory (score matching) and practice both favor the noise target._

**Answer:** They are interchangeable in principle (Eq. 4 relates them), but predicting $\epsilon$ gives a
                 fixed-scale, well-conditioned target &mdash; standard Gaussian noise at every timestep
                 &mdash; and connects the loss to denoising score matching. The paper found this
                 parameterization (Eq. 11) plus the unweighted MSE (Eq. 14) simply trains better and produces
                 higher-quality samples.

</details>